In [31]:
import gradio as gr
from openai import OpenAI
import json
from rake_nltk import Rake

In [32]:
openai = OpenAI(base_url="http://localhost:11434/v1", api_key = "ollama")

In [33]:
system_prompt = """You are very helpful assistant"""

In [34]:
def extract_keywords(sentence):
    """For given sentence extract keyword """
    rake = Rake()

    rake.extract_keywords_from_text(sentence)

    keywords = rake.get_ranked_phrases()
    if keywords:
        return keywords
    return "No Keywords"



In [35]:
tool = [
    {
        "type": "function",
        "function": {
            "name": "extract_keywords",
            "description": "Extract Keywords from given sentence",
            "parameters": {
                "type": "object",
                "properties": {
                    "sentence": {
                        "type": "string"
                    }
                },
                "required": ["sentence"]
            }
        }
    }]

In [36]:
tools = tool  # already correctly structured

In [37]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "extract_keywords":  # fixed: was "extract_keyword"
        arguments = json.loads(tool_call.function.arguments)
        get_sentence = arguments.get("sentence")
        extracted_keywords = extract_keywords(get_sentence)
        response = {
            "role": "tool",
            "content": str(extracted_keywords),  # fixed: was the function reference
            "tool_call_id": tool_call.id
        }
    return response

In [38]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content" : system_prompt}] + history + [{"role":"user","content": message}]
    response = openai.chat.completions.create(model = "devstral:24b", messages= messages, tools = tools)
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model = "devstral:24b", messages=messages)
    return response.choices[0].message.content


In [39]:
gr.ChatInterface(fn = chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/apoorva/Desktop/example/code_translation/tools/lib/python3.10/site-packages/gradio/queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
  File "/home/apoorva/Desktop/example/code_translation/tools/lib/python3.10/site-packages/gradio/route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
  File "/home/apoorva/Desktop/example/code_translation/tools/lib/python3.10/site-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
  File "/home/apoorva/Desktop/example/code_translation/tools/lib/python3.10/site-packages/gradio/blocks.py", line 1634, in call_function
    prediction = await fn(*processed_input)
  File "/home/apoorva/Desktop/example/code_translation/tools/lib/python3.10/site-packages/gradio/utils.py", line 1027, in async_wrapper
    response = await f(*args, **kwargs)
  File "/home/apoorva/Desktop/example/code_translat